In [14]:
# ════════════════════════════════════════════════════════════
# Cell 1 — Imports
# ════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from tqdm import tqdm

In [2]:
# ════════════════════════════════════════════════════════════
# Cell 2 — Attention
# ════════════════════════════════════════════════════════════
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    score = Q @ K.transpose(-2, -1) / d_k ** 0.5
    if mask is not None:
        score = score.masked_fill(mask, -1e9)
    score = F.softmax(score, dim=-1)
    return score @ V


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, d_k=64, d_v=64, head=8):
        super().__init__()
        self.nhead = head
        self.d_k   = d_k
        self.d_v   = d_v
        self.W_Q = nn.Linear(d_model, head * d_k)
        self.W_K = nn.Linear(d_model, head * d_k)
        self.W_V = nn.Linear(d_model, head * d_v)
        self.W_O = nn.Linear(head * d_v, d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.W_Q(Q)
        K = self.W_K(K)
        V = self.W_V(V)
        B, S,   _ = Q.size()
        _, S_K, _ = K.size()
        Q = Q.view(B, S,   self.nhead, self.d_k).transpose(1, 2)
        K = K.view(B, S_K, self.nhead, self.d_k).transpose(1, 2)
        V = V.view(B, S_K, self.nhead, self.d_v).transpose(1, 2)
        O = scaled_dot_product_attention(Q, K, V, mask)
        O = O.transpose(1, 2).reshape(B, S, self.nhead * self.d_v)
        return self.W_O(O)


In [3]:
def pos_encoder(emb, max_len, d_model):
    # emb: max_len x d_model
    
    pos = torch.arange(max_len).unsqueeze(1) # max_len x 1
    i = torch.arange(0, d_model, 2)
    div = 10000 ** (i / d_model)

    emb[:, 0::2] = torch.sin(pos / div)
    emb[:, 1::2] = torch.cos(pos / div)

    return emb

class EncoderBlock(nn.Module):
    def __init__(self, d_model, nhead, d_ff, dropout):
        super().__init__()
        self.attn = MultiHeadAttention(d_model=d_model, head=nhead)
        self.ff1 = nn.Linear(d_model, d_ff)
        self.ff2 = nn.Linear(d_ff, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.act = nn.ReLU()

        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, pad_mask):
        x2 = self.attn(x, x, x, pad_mask)
        x = x + self.dropout(x2)
        x = self.ln1(x)

        x2 = self.ff2(self.act(self.ff1(x)))
        x = x + self.dropout(x2)
        x = self.ln2(x)
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=512, nhead=8, num_layers=6, d_ff=2048, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, nhead, d_ff, dropout) for _ in range(num_layers)
        ])
        pos = torch.zeros(max_len, d_model)
        pos = pos_encoder(pos, max_len, d_model)
        self.register_buffer("pos", pos)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, pad_mask):
        x = self.dropout(self.embedding(x) + self.pos[:x.size(1), :])
        for block in self.blocks:
            x = block(x, pad_mask)
        
        return x

class DecoderBlock(nn.Module):
    def __init__(self, max_len, d_model, nhead, d_ff, dropout):
        super().__init__()
        self.attn1 = MultiHeadAttention(d_model=d_model, head=nhead)
        self.attn2 = MultiHeadAttention(d_model=d_model, head=nhead)
        self.ff1 = nn.Linear(d_model, d_ff)
        self.ff2 = nn.Linear(d_ff, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)
        self.act = nn.ReLU()

        self.register_buffer("mask", torch.triu(torch.ones(max_len, max_len), diagonal=1).bool())
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, y, src_mask, tgt_mask):
        # x: batch_size x seq_len

        comb = self.mask[:x.size(1), :x.size(1)] | tgt_mask
        # masked multi-head attention
        x2 = self.attn1(x, x, x, comb)
        x = x + self.dropout(x2)
        x = self.ln1(x)

        # cross-attention
        x2 = self.attn2(x, y, y, src_mask)
        x = x + self.dropout(x2)
        x = self.ln2(x)

        # ff layer
        x2 = self.ff2(self.act(self.ff1(x)))
        x = x + self.dropout(x2)
        x = self.ln3(x)
            
        return x

class Decoder(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=512, nhead=8, num_layers=6, d_ff=2048, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            DecoderBlock(max_len, d_model, nhead, d_ff, dropout) for _ in range(num_layers)
        ])

        pos = torch.zeros(max_len, d_model)
        pos = pos_encoder(pos, max_len, d_model)
        self.register_buffer("pos", pos)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, y, src_mask, tgt_mask):
        x = self.dropout(self.embedding(x) + self.pos[:x.size(1), :])
        for block in self.blocks:
            x = block(x, y, src_mask, tgt_mask)
        return x

class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, max_len, d_model=512, nhead=8, num_layers=6, d_ff=2048, dropout=0.1):
        super().__init__()
        self.enc = Encoder(src_vocab, max_len, d_model, nhead, num_layers, d_ff, dropout)
        self.dec = Decoder(tgt_vocab, max_len, d_model, nhead, num_layers, d_ff, dropout)
        self.fc = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt, src_mask, tgt_mask):
        enc_out = self.enc(src, src_mask)
        dec_out = self.dec(tgt, enc_out, src_mask, tgt_mask)
        return self.fc(dec_out)

In [4]:
# ════════════════════════════════════════════════════════════
# Cell 4 — 데이터 로드 (WMT19 de-en, validation only)
# ════════════════════════════════════════════════════════════
train_raw = load_dataset("wmt19", "de-en", split="train[:50000]")
test = load_dataset("wmt19", "de-en", split="validation")
split = test.train_test_split(test_size=0.5, seed=42)
dev_raw = split["train"]
test_raw  = split["test"]

print(f"train: {len(train_raw)}문장, test: {len(test_raw)}문장")
print("sample:", train_raw[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

de-en/train-00000-of-00016.parquet:   0%|          | 0.00/384M [00:00<?, ?B/s]

de-en/train-00001-of-00016.parquet:   0%|          | 0.00/130M [00:00<?, ?B/s]

de-en/train-00002-of-00016.parquet:   0%|          | 0.00/102M [00:00<?, ?B/s]

...

de-en/train-00015-of-00016.parquet:   0%|          | 0.00/231M [00:00<?, ?B/s]

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/495k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/34782245 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2998 [00:00<?, ? examples/s]

train: 50000문장, test: 1499문장
sample: {'translation': {'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}}


In [5]:
def load_tokenizer():
    tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
    return tok

In [6]:
def encode_data(data, tok):
    X, Y_input, Y_output = [], [], []
    for item in data:
        src = item['translation']['de']
        tgt = item['translation']['en']

        X.append(tok.encode(src))
        Y = tok.encode(tgt)
        Y_input.append(Y[:-1])
        Y_output.append(Y[1:])
    return X, Y_input, Y_output

In [7]:
tok = load_tokenizer()
X, Y_input, Y_output = encode_data(train_raw, tok)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
class TransDataset(Dataset):
    def __init__(self, X, Y_input, Y_output):
        self.X = X
        self.Y_input = Y_input
        self.Y_output = Y_output
    
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y_input[idx], self.Y_output[idx]

In [9]:
PAD_ID = tok.pad_token_id

def collate_fn(batch):
    x, y_in, y_out = zip(*batch)

    x = pad_sequence([torch.tensor(s) for s in x], batch_first=True, padding_value=PAD_ID)
    y_in = pad_sequence([torch.tensor(s) for s in y_in], batch_first=True, padding_value=PAD_ID)
    y_out = pad_sequence([torch.tensor(s) for s in y_out], batch_first=True, padding_value=PAD_ID)

    return x, y_in, y_out

In [10]:
def make_pad_mask(seq, pad_id):
    # seq: batch x seq_len
    # return: batch x 1 x 1 x seq_len
    return (seq == pad_id).unsqueeze(1).unsqueeze(2)

In [11]:
tok = load_tokenizer()
PAD_ID = tok.pad_token_id

# training set
train_X, train_Y_in, train_Y_out = encode_data(train_raw, tok)
trainset = TransDataset(train_X, train_Y_in, train_Y_out)
trainloader = DataLoader(trainset, batch_size=32, shuffle=True, collate_fn=collate_fn)


In [12]:
# src_vocab, tgt_vocab, max_len, d_model=512, nhead=8, num_layers=6, d_ff=2048, dropout=0.1
VOCAB_SIZE = tok.vocab_size
MAX_LEN = max(len(item)for item in train_X)
D_MODEL = 512
NHEAD = 8
NUM_LAYERS = 6
D_FF = 2048
DROPOUT = 0.1

learning_rate = 0.001
num_epochs = 3

model = Transformer(
    src_vocab = VOCAB_SIZE, tgt_vocab = VOCAB_SIZE, max_len = MAX_LEN, d_model = D_MODEL, nhead = NHEAD, num_layers = NUM_LAYERS, d_ff = D_FF, dropout = DROPOUT
)

In [13]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [15]:
dev_X, dev_Y_in, dev_Y_out = encode_data(dev_raw, tok)
devset = TransDataset(dev_X, dev_Y_in, dev_Y_out)
devloader = DataLoader(devset, batch_size=32, shuffle=True, collate_fn=collate_fn)


In [ ]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for x, y_in, y_out in tqdm(trainloader):
        # Forward pass
        src_mask = make_pad_mask(x, PAD_ID)
        tgt_mask = make_pad_mask(y_in, PAD_ID)
        
        outputs = model(x, y_in, src_mask, tgt_mask) # (batch_size, seq_len, vocab_size)
        outputs = outputs.view(-1, VOCAB_SIZE) # (batch_size * seq_len, vocab_size)
        y_out = y_out.view(-1)
        loss = criterion(outputs, y_out)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(trainloader) 
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

    # evaluate validation loss
    eval_loss = 0
    model.eval()
    with torch.no_grad():
        for x, y_in, y_out in tqdm(devloader):
            # Forward pass
            src_mask = make_pad_mask(x, PAD_ID)
            tgt_mask = make_pad_mask(y_in, PAD_ID)
            
            outputs = model(x, y_in, src_mask, tgt_mask) # (batch_size, seq_len, vocab_size)
            outputs = outputs.view(-1, VOCAB_SIZE) # (batch_size * seq_len, vocab_size)
            y_out = y_out.view(-1)
            loss = criterion(outputs, y_out)

            # Backward pass
            optimizer.zero_grad()
            optimizer.step()

            eval_loss += loss.item()
    avg_eval_loss = eval_loss / len(devloader) 
    print(f'Epoch [{epoch+1}/{num_epochs}], Validation Loss: {avg_eval_loss:.4f}')


: 

: 

: 

: 

In [ ]:
!python main.py

====Data succesfully loaded!====
====Tokenizer succesfully loaded!====
====Trainset, Devset loaded successfully!====
Start Training...
100% 625/625 [01:24<00:00,  7.43it/s]
Epoch [1/5], Loss: 3965926440108.0518
100% 94/94 [00:02<00:00, 33.45it/s]
Epoch [1/5], Validation Loss: 5852572574218.8936
 60% 377/625 [00:49<00:34,  7.09it/s]

 61% 379/625 [00:50<00:34,  7.14it/s]

: 

====Data succesfully loaded!====
====Tokenizer succesfully loaded!====
====Trainset, Devset loaded successfully!====
Start Training...
 15% 94/625 [00:12<01:04,  8.19it/s]